# Scrublet Doublet Detection

Run scrublet per batch on the combined RNA h5ad. Adds `doublet_score` and `predicted_doublet` columns.

**Input:** `results/immune_integration/adata_rna.h5ad` (from Notebook 1)
**Output:** Same file, updated with doublet scores.

In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import scanpy as sc
from matplotlib import rcParams

# Add notebook directory to path for data_loading_utils (if needed)
sys.path.insert(0, os.path.join(os.getcwd(), "docs", "notebooks", "immune_integration"))

rcParams["pdf.fonttype"] = 42

In [ ]:
results_folder = "/nfs/team205/vk7/sanger_projects/my_packages/regularizedvi/results/immune_integration/"
rna_path = os.path.join(results_folder, "adata_rna.h5ad")
adata = sc.read_h5ad(rna_path)
print(f"Loaded: {adata.shape}")
print(f"Batches: {adata.obs['batch'].nunique()}")

## Run scrublet per batch

In [ ]:
MIN_CELLS_SCRUBLET = 50
adata.obs["doublet_score"] = np.nan
adata.obs["predicted_doublet"] = False
batch_counts = adata.obs["batch"].value_counts()
skipped = []

for batch_name in batch_counts.index:
    mask = adata.obs["batch"] == batch_name
    n_cells = mask.sum()
    if n_cells < MIN_CELLS_SCRUBLET:
        skipped.append((batch_name, n_cells))
        continue
    adata_batch = adata[mask].copy()
    n_comps = min(30, n_cells - 1)
    sc.pp.scrublet(adata_batch, threshold=0.25, n_prin_comps=n_comps)
    adata.obs.loc[mask, "doublet_score"] = adata_batch.obs["doublet_score"].values
    adata.obs.loc[mask, "predicted_doublet"] = adata_batch.obs["predicted_doublet"].values
    del adata_batch

if skipped:
    print(f"Skipped {len(skipped)} batches with < {MIN_CELLS_SCRUBLET} cells: {skipped}")
n_doublets = adata.obs["predicted_doublet"].sum()
print(f"Predicted doublets: {n_doublets} / {adata.n_obs} ({100 * n_doublets / adata.n_obs:.1f}%)")

## Doublet score distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Overall distribution
axes[0].hist(adata.obs["doublet_score"].dropna(), bins=100, edgecolor="none")
axes[0].set_xlabel("Doublet score")
axes[0].set_ylabel("Cells")
axes[0].set_title("Overall doublet score distribution")
axes[0].axvline(0.25, color="red", linestyle="--", label="threshold=0.25")
axes[0].legend()

# Per-dataset
for ds in sorted(adata.obs["dataset"].unique()):
    scores = adata.obs.loc[adata.obs["dataset"] == ds, "doublet_score"].dropna()
    axes[1].hist(scores, bins=50, alpha=0.5, label=ds, edgecolor="none")
axes[1].set_xlabel("Doublet score")
axes[1].set_ylabel("Cells")
axes[1].set_title("Doublet score by dataset")
axes[1].legend(fontsize=7)

plt.tight_layout()
plt.show()

## Save updated h5ad

In [ ]:
adata.write_h5ad(rna_path)
print(f"Saved: {rna_path}")
print(f"  Shape: {adata.shape}")
print("  Doublet columns: doublet_score, predicted_doublet")

# Summary per dataset
for ds in sorted(adata.obs["dataset"].unique()):
    mask = adata.obs["dataset"] == ds
    n = mask.sum()
    n_d = adata.obs.loc[mask, "predicted_doublet"].sum()
    print(f"  {ds}: {n_d}/{n} doublets ({100 * n_d / n:.1f}%)")